# Logic Programming with Prolog

This notebook covers writing Prolog programs to `.pl` files with `%%writefile` and executing them from notebook code cells. Topics covered include facts, rules, variables, unification, backtracking, recursion, arithmetic, CUT, and negation as failure.

### Learning outcomes

By the end of this notebook, you should be able to:

- Represent knowledge using facts and rules;
- Ask ground and variable-based queries;
- Explain unification and the order of answers;
- Trace simple backtracking behavior;
- Define recursive predicates over relationships and lists;
- Use arithmetic comparisons and `is/2` correctly;
- Explain the basic effects of CUT and negation as failure; and
- Combine these ideas in a small route-finding program.

---

## Suggested workflow

- Write Prolog source files with `%%writefile`.
- Run the `swipl` CLI runner cell.
- Review results and outputs.

---

# 0. Environment setup

Verification of **SWI-Prolog** availability in the notebook environment.

In [ ]:
# Check that the SWI-Prolog executable is available to this notebook.
import shutil

swipl_path = shutil.which("swipl")
if swipl_path is None:
    print("SWI-Prolog was not found. Follow the installation instructions above.")
else:
    print("SWI-Prolog found at:", swipl_path)

SWI-Prolog found at: /usr/bin/swipl


In [ ]:
# Google Colab only: remove the leading # characters if SWI-Prolog is missing.
!apt-get update -qq
!apt-get install -y -qq swi-prolog-nox

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package gettext-base.
(Reading database ... 122809 files and directories currently installed.)
Preparing to unpack .../00-gettext-base_0.21-14ubuntu2_amd64.deb ...
Unpacking gettext-base (0.21-14ubuntu2) ...
Selecting previously unselected package autopoint.
Preparing to unpack .../01-autopoint_0.21-14ubuntu2_all.deb ...
Unpacking autopoint (0.21-14ubuntu2) ...
Selecting previously unselected package libdebhelper-perl.
Preparing to unpack .../02-libdebhelper-perl_13.14.1ubuntu5_all.deb ...
Unpacking libdebhelper-perl (13.14.1ubuntu5) ...
Selecting previously unselected package libtool.
Preparing to unpack .../03-libtool_2.4.7-7build1_all.deb ...
Unpacking libtool (2.4.7-7build1) ...
Selecting previously unselected package dh-autoreconf.
Preparing to unpack .../04-dh-au

In [ ]:
!swipl --version

SWI-Prolog version 9.0.4 for x86_64-linux


The command used throughout the notebook has this form:

```text
swipl -q -s program.pl -g run -t halt
```

- `-q` starts quietly.
- `-s program.pl` loads the source file.
- `-g run` calls the goal `run`.
- `-t halt` exits when the goal is finished.

---
# 1. Facts, rules, and first queries

A **fact** states that a relationship is true. A **rule** states that a conclusion is true when its body is true. Atoms normally begin with lowercase letters; variables begin with uppercase letters or `_`. Every fact and rule ends with a period.

In [ ]:
%%writefile 01_basics.pl
% Facts describe courses, students, and study relationships.
course(logic).
course(ai).

student(mai).
student(lan).
student(noah).

studies(mai, logic).
studies(noah, logic).
studies(lan, ai).
studies(lan, logic).

% A person is learning logic when that person studies logic.
learning(Person) :-
    student(Person),
    studies(Person, logic).

% This runner prints one ground result and all variable bindings.
run :-
    learning(mai),
    writeln('learning(mai): true'),
    findall(Person, learning(Person), People),
    format('learning(Person): ~w~n', [People]).

Writing 01_basics.pl


In [ ]:
!swipl -q -s 01_basics.pl -g run -t halt
!swipl -q -s 01_basics.pl -g "(learning(lan) -> writeln('learning(lan): true') ; writeln('learning(lan): false'))" -t halt

learning(mai): true
learning(Person): [mai,lan,noah]
learning(lan): true


### Section 1 Observations

- **Fact definition:** Adding new facts directly expands the knowledge base of Prolog, allowing previously failing queries to resolve successfully.
- **Rule evaluation:** Rules are dependent on both student declarations and study relations matching the goals.

In [ ]:
%%writefile 01_basics_updated.pl
% Facts describe courses, students, and study relationships.
course(logic).
course(ai).

student(mai).
student(lan).
student(noah).

studies(mai, logic).
studies(noah, logic).
studies(lan, ai).
studies(lan, logic).

% A person is learning logic when that person studies logic.
learning(Person) :-
    student(Person),
    studies(Person, logic).

Writing 01_basics_updated.pl


In [ ]:
!swipl -q -s 01_basics_updated.pl -g "(learning(lan) -> writeln('learning(lan): true') ; writeln('learning(lan): false'))" -t halt

learning(lan): true


---
# 2. Terms and unification

**Unification** makes two terms identical by finding variable bindings. Compound terms unify only when their functors and arities match and their corresponding arguments can unify. Lists are also terms: `[Head|Tail]` separates the first element from the remaining list.

In [ ]:
%%writefile 02_unification.pl
% The runner demonstrates successful and failed unification.
run :-
    person(Name, Age) = person(ada, 20),
    format('Name = ~w, Age = ~w~n', [Name, Age]),

    [Head|Tail] = [logic, prolog, ai],
    format('Head = ~w, Tail = ~w~n', [Head, Tail]),

    f(X, a) = f(b, Y),
    format('X = ~w, Y = ~w~n', [X, Y]),

    (f(_) = g(_) -> Result = succeeds ; Result = fails),
    format('f(_) = g(_) : ~w~n', [Result]).

Writing 02_unification.pl


In [ ]:
!swipl -q -s 02_unification.pl -g run -t halt

Name = ada, Age = 20
Head = logic, Tail = [prolog,ai]
X = b, Y = a
f(_) = g(_) : fails


### Section 2 Observations

- **Arity and Functors:** Terms unify successfully when they share identical functor names and arities, allowing corresponding argument variables to bind.
- **Unification failure:** Discrepant functors prevent unification regardless of argument symmetry.
- **List pattern matching:** The `[A, B|Rest]` pattern decomposes a list by binding the first two items and putting the remaining elements in a sublist.

In [ ]:
%%writefile 02_unification_updated.pl
% The runner demonstrates successful and failed unification.
run :-
    person(Name, Age) = person(ada, 20),
    format('Name = ~w, Age = ~w~n', [Name, Age]),

    [Head|Tail] = [logic, prolog, ai],
    format('Head = ~w, Tail = ~w~n', [Head, Tail]),

    f(X, a) = f(b, Y),
    format('X = ~w, Y = ~w~n', [X, Y]),

    [A, B|Rest] = [1, 2, 3, 4],
    format('A = ~w, B = ~w, Rest = ~w~n', [A, B, Rest]),

    (f(_) = g(_) -> Result = succeeds ; Result = fails),
    format('f(_) = g(_) : ~w~n', [Result]).

Writing 02_unification_updated.pl


In [ ]:
!swipl -q -s 02_unification_updated.pl -g run -t halt

Name = ada, Age = 20
Head = logic, Tail = [prolog,ai]
X = b, Y = a
A = 1, B = 2, Rest = [3,4]
f(_) = g(_) : fails


---
# 3. Backtracking and answer order

Prolog selects the leftmost subgoal, tries clauses from top to bottom, and searches depth first. When a later subgoal fails, Prolog returns to the most recent choice point and tries another alternative.

In [ ]:
%%writefile 03_backtracking.pl
% Candidate order determines the order in which Prolog explores meals.
candidate(salad).
candidate(burger).
candidate(curry).
candidate(soup).

vegetarian(salad).
vegetarian(curry).
vegetarian(soup).

healthy(salad).
healthy(soup).

% Both subgoals must succeed for the same Meal binding.
suitable(Meal) :-
    candidate(Meal),
    vegetarian(Meal),
    healthy(Meal).

run :-
    findall(Meal, suitable(Meal), Meals),
    format('Suitable meals in answer order: ~w~n', [Meals]).

Writing 03_backtracking.pl


In [ ]:
!swipl -q -s 03_backtracking.pl -g run -t halt

Suitable meals in answer order: [salad,soup]


### Section 3 Observations

- **Search order:** The ordering of facts in the source database directly dictates the sequence of results generated via backtracking.
- **Subgoal evaluation:** Reordering subgoals within a rule alters the operational evaluation path while matching the same logical constraints.

In [ ]:
%%writefile 03_backtracking_reordered_fact.pl
% Candidate order determines the order in which Prolog explores meals.
candidate(soup).
candidate(burger).
candidate(curry).
candidate(salad).

vegetarian(salad).
vegetarian(curry).
vegetarian(soup).

healthy(salad).
healthy(soup).

% Both subgoals must succeed for the same Meal binding.
suitable(Meal) :-
    healthy(Meal),
    vegetarian(Meal),
    candidate(Meal).

run :-
    findall(Meal, suitable(Meal), Meals),
    format('Suitable meals in answer order: ~w~n', [Meals]).

Writing 03_backtracking_reordered_fact.pl


In [ ]:
!swipl -q -s 03_backtracking_reordered_fact.pl -g run -t halt

Suitable meals in answer order: [salad,soup]


In [ ]:
%%writefile 03_backtracking_reordered_subgoals.pl
% Candidate order determines the order in which Prolog explores meals.
candidate(salad).
candidate(burger).
candidate(curry).
candidate(soup).

vegetarian(salad).
vegetarian(curry).
vegetarian(soup).

healthy(salad).
healthy(soup).

% Both subgoals must succeed for the same Meal binding.
suitable(Meal) :-
    healthy(Meal),
    vegetarian(Meal),
    candidate(Meal).

run :-
    findall(Meal, suitable(Meal), Meals),
    format('Suitable meals in answer order: ~w~n', [Meals]).

Writing 03_backtracking_reordered_subgoals.pl


In [ ]:
!swipl -q -s 03_backtracking_reordered_subgoals.pl -g run -t halt

Suitable meals in answer order: [salad,soup]


---
# 4. Building recursive relationships

A recursive definition needs:

- a **base case** that can succeed directly; and
- a **recursive case** that reduces the problem to a smaller or closer subproblem.

In [ ]:
%%writefile 04_family.pl
% Direct parent relationships form an acyclic family graph.
parent(alex, beth).
parent(alex, dan).
parent(beth, claire).
parent(claire, emma).

grandparent(X, Y) :-
    parent(X, Middle),
    parent(Middle, Y).

% Base case: a parent is an ancestor.
ancestor(X, Y) :-
    parent(X, Y).

% Recursive case: move one generation toward Y.
ancestor(X, Y) :-
    parent(X, Middle),
    ancestor(Middle, Y).

run :-
    findall(Person, ancestor(alex, Person), Descendants),
    format('Descendants of alex: ~w~n', [Descendants]),
    findall(Person, grandparent(Person, claire), Grandparents),
    format('Grandparents of claire: ~w~n', [Grandparents]).

Writing 04_family.pl


In [ ]:
!swipl -q -s 04_family.pl -g run -t halt

Descendants of alex: [beth,dan,claire,emma]
Grandparents of claire: [alex]


### Section 4 Observations

- **Acyclic graphs:** Basic recursive rules navigate directed acyclic graphs to resolve ancestor networks sequentially.
- **Sibling rules:** Sibling relationships can be established using mutual parent relationships coupled with inequality constraints.
- **Cycle risks:** Recursive path traversals on cyclic graphs can trigger infinite loops if visited checks or cycle prevention are omitted.

In [ ]:
%%writefile 04_family_updated.pl
% Direct parent relationships form an acyclic family graph.
parent(alex, beth).
parent(alex, dan).
parent(beth, claire).
parent(claire, emma).
parent(alex, eva).

grandparent(X, Y) :-
    parent(X, Middle),
    parent(Middle, Y).

% Base case: a parent is an ancestor.
ancestor(X, Y) :-
    parent(X, Y).

% Recursive case: move one generation toward Y.
ancestor(X, Y) :-
    parent(X, Middle),
    ancestor(Middle, Y).

sibling(Person1, Person2) :-
    parent(P, Person1),
    parent(P, Person2),
    Person1 \= Person2.

run :-
    findall(Person, ancestor(alex, Person), Descendants),
    format('Descendants of alex: ~w~n', [Descendants]),
    findall(Person, grandparent(Person, claire), Grandparents),
    format('Grandparents of claire: ~w~n', [Grandparents]),
    findall(Sibling, sibling(dan, Sibling), DansSiblings),
    format('Siblings of dan: ~w~n', [DansSiblings]).

Writing 04_family_updated.pl


In [ ]:
!swipl -q -s 04_family_updated.pl -g run -t halt

Descendants of alex: [beth,dan,eva,claire,emma]
Grandparents of claire: [alex]
Siblings of dan: [beth,eva]


---
# 5. Lists and structural recursion

A non-empty list has the form `[Head|Tail]`. Recursive list predicates process `Head` and then call themselves on the shorter `Tail`.

In [ ]:
%%writefile 05_lists_guided.pl
% Membership succeeds at the head or recursively in the tail.
member_of(X, [X|_]).
member_of(X, [_|Tail]) :-
    member_of(X, Tail).

% The empty list has length zero.
list_length([], 0).
list_length([_|Tail], Length) :-
    list_length(Tail, TailLength),
    Length is TailLength + 1.

% Summation uses the same base-and-recursive-case pattern.
sum_numbers([], 0).
sum_numbers([Head|Tail], Sum) :-
    sum_numbers(Tail, TailSum),
    Sum is Head + TailSum.

run :-
    findall(X, member_of(X, [logic, prolog, ai]), Members),
    format('Members: ~w~n', [Members]),
    list_length([logic, prolog, ai], Length),
    format('Length: ~w~n', [Length]),
    sum_numbers([3, 5, 2, 10], Sum),
    format('Sum: ~w~n', [Sum]).

Writing 05_lists_guided.pl


In [ ]:
!swipl -q -s 05_lists_guided.pl -g run -t halt

Members: [logic,prolog,ai]
Length: 3
Sum: 20


### List exercise

Replace both `fail` stubs below. Do not use the built-in `last/2`, `include/3`, or `aggregate/3`.

1. `last_item(List, Item)` returns the final item of a non-empty list.
2. `count_item(Item, List, Count)` counts occurrences of `Item`.

Hints:

- The one-element list `[X]` is a useful base case for `last_item/2`.
- `count_item/3` needs a base case for `[]`, one case where the head matches, and one where it does not.
- Use `\=/2` in the non-matching clause.

In [ ]:
%%writefile 05_lists_exercise.pl
% last_item(List, Item) returns the final item of a non-empty list.
last_item([Item], Item).
last_item([_|Tail], Item) :-
    last_item(Tail, Item).

% count_item(Item, List, Count) counts occurrences of Item.
count_item(_, [], 0).
count_item(Item, [Item|Tail], Count) :-
    count_item(Item, Tail, TailCount),
    Count is TailCount + 1.
count_item(Item, [Head|Tail], Count) :-
    Item \= Head,
    count_item(Item, Tail, Count).

% The tests return empty lists until the predicates are implemented.
run :-
    findall(X, last_item([logic, is, powerful], X), LastAnswers),
    format('last_item answers: ~w~n', [LastAnswers]),
    findall(N, count_item(a, [a, b, a, c, a], N), CountAnswers),
    format('count_item answers: ~w~n', [CountAnswers]).

Writing 05_lists_exercise.pl


In [ ]:
!swipl -q -s 05_lists_exercise.pl -g run -t halt

last_item answers: [powerful]
count_item answers: [3]


### Section 5 Observations

- **Structural reduction:** Recursive list operations decrement the problem size on each invocation by traversing the tail of the list, ensuring termination at the empty list base case.

---
# 6. Arithmetic and numeric conditions

In Prolog, `=` performs unification; it does not evaluate arithmetic. Use `is/2` to evaluate an expression, and use comparisons such as `>`, `>=`, `<`, `=<`, `=:=`, and `=
=` for numbers.

In [ ]:
%%writefile 06_arithmetic.pl
% The conditions are mutually exclusive.
result_label(Score, distinction) :-
    Score >= 85.
result_label(Score, pass) :-
    Score >= 50,
    Score < 85.
result_label(Score, fail) :-
    Score < 50.

% The denominator must be non-zero before division.
average(Total, Count, Average) :-
    Count > 0,
    Average is Total / Count.

run :-
    findall(Label, result_label(72, Label), Labels),
    format('Label for 72: ~w~n', [Labels]),
    average(240, 3, Average),
    format('Average: ~2f~n', [Average]).

Writing 06_arithmetic.pl


In [ ]:
!swipl -q -s 06_arithmetic.pl -g run -t halt

Label for 72: [pass]
Average: 80.00


### Arithmetic exercise: factorial

Replace the stub with a base case and a recursive case for non-negative integers. For `N > 0`, calculate `N1 is N - 1`, recursively compute `factorial(N1, F1)`, and then calculate `F is N * F1`.

In [ ]:
%%writefile 06_factorial_exercise.pl
% TODO: replace this stub with the factorial definition.
factorial(0, 1).
factorial(N, F) :-
    N > 0,
    N1 is N - 1,
    factorial(N1, F1),
    F is N * F1.


run :-
    findall(F0, factorial(0, F0), BaseAnswers),
    findall(F5, factorial(5, F5), RecursiveAnswers),
    format('factorial(0, F): ~w~n', [BaseAnswers]),
    format('factorial(5, F): ~w~n', [RecursiveAnswers]).

Writing 06_factorial_exercise.pl


In [ ]:
!swipl -q -s 06_factorial_exercise.pl -g run -t halt

factorial(0, F): [1]
factorial(5, F): [120]


### Section 6 Observations

- **Arithmetic evaluation:** The `is/2` operator acts as the designated mechanism for processing numeric computations in Prolog.
- **Precondition checks:** Boundary validation in recursive math functions prevents infinite loops and ensures proper termination.

---
# 7. Basic control: CUT and negation as failure

## 7.1 CUT

The CUT symbol `!` commits to choices made since the current predicate invocation began. It can make a predicate deterministic, but it also prunes alternative answers.

In [ ]:
%%writefile 07_cut.pl
% Options are considered from top to bottom.
option(bus).
option(train).
option(bike).

available(train).
available(bike).

all_available(Option) :-
    option(Option),
    available(Option).

% CUT keeps only the first available option found in program order.
first_available(Option) :-
    option(Option),
    available(Option),
    !.

run :-
    findall(X, all_available(X), All),
    findall(X, first_available(X), First),
    format('All available: ~w~n', [All]),
    format('First available: ~w~n', [First]).

Writing 07_cut.pl


In [ ]:
!swipl -q -s 07_cut.pl -g run -t halt

All available: [train,bike]
First available: [train]


### Section 7.1 Observations

- **Pruning choices:** The use of CUT restricts search options to the first successful match in database ordering.
- **Red CUT effects:** Introducing CUT can change the logical result set by dropping valid solutions.

## 7.2 Negation as failure

The goal `\+ Goal` succeeds when Prolog cannot prove `Goal`. It is most reliable when the relevant facts are complete under a closed-world assumption and variables in the negative goal are already bound.

In [ ]:
%%writefile 07_negation_before.pl
registered(mai).
registered(noah).
suspended(noah).

% registered/1 binds Person before the negative check.
eligible(Person) :-
    registered(Person),
    \+ suspended(Person).

run :-
    findall(Person, eligible(Person), Eligible),
    format('Eligible before the new fact: ~w~n', [Eligible]),
    findall(Person, (\+ suspended(Person), registered(Person)), WrongOrder),
    format('Negative goal first: ~w~n', [WrongOrder]).

Writing 07_negation_before.pl


In [ ]:
!swipl -q -s 07_negation_before.pl -g run -t halt

Eligible before the new fact: [mai]
Negative goal first: []


Now add one fact and rerun the same logical rule.

In [ ]:
%%writefile 07_negation_after.pl
registered(mai).
registered(noah).
suspended(noah).
suspended(mai).

eligible(Person) :-
    registered(Person),
    \+ suspended(Person).

run :-
    findall(Person, eligible(Person), Eligible),
    format('Eligible after the new fact: ~w~n', [Eligible]).

Writing 07_negation_after.pl


In [ ]:
!swipl -q -s 07_negation_after.pl -g run -t halt

Eligible after the new fact: []


### Section 7.2 Observations

- **Non-monotonic reasoning:** Adding new negative facts can retract previously derived assertions.
- **Closed-world assumption:** Negation as failure operates on the assumption that anything unprovable is false.
- **Binding ordering:** Negative checks are placed after bound goals to prevent premature evaluation failures on unbound variables.

---
# 8. Integrated challenge: route finder

A pathfinding program on a directed acyclic network using `road(From, To, Cost)` predicates.

`route(From, To, Path, Cost)` utilizes two logical cases:

1. **Direct route:** uses a direct road relationship.
2. **Indirect route:** recursively traverses an adjacent node, increments the path weight, and appends the source to the remaining path.

In [ ]:
%%writefile 08_route_challenge.pl
% Directed roads with positive travel costs.
road(campus, library, 2).
road(campus, cafe, 1).
road(library, museum, 4).
road(cafe, park, 2).
road(park, museum, 2).
road(cafe, museum, 6).

% Direct route: a road directly connects From to To; the path is [From, To].
route(From, To, [From, To], Cost) :-
    road(From, To, Cost).

% Indirect route: a road connects From to Next; recursively find a route from Next to To;
% add the first road cost to the remaining cost; and build the path as [From|RestPath].
route(From, To, [From|RestPath], TotalCost) :-
    road(From, Next, RoadCost),
    route(Next, To, RestPath, RestCost),
    TotalCost is RoadCost + RestCost.

% Print every route in Prolog's search order.
run :-
    findall(Path-Cost, route(campus, museum, Path, Cost), Routes),
    format('Routes from campus to museum: ~w~n', [Routes]).

Overwriting 08_route_challenge.pl


In [ ]:
!swipl -q -s 08_route_challenge.pl -g run -t halt

Routes from campus to museum: [[campus,library,museum]-6,[campus,cafe,museum]-7,[campus,cafe,park,museum]-5]


### Challenge tests

After completing `route/4`, both tests should pass.

In [ ]:
!swipl -q -s 08_route_challenge.pl -g "(route(campus,museum,[campus,cafe,park,museum],5) -> writeln('Test 1 passed') ; writeln('Test 1 failed'))" -t halt
!swipl -q -s 08_route_challenge.pl -g "findall(P,route(campus,museum,P,_),Ps),length(Ps,N),format('Number of routes: ~w~n',[N])" -t halt

Test 1 passed
Number of routes: 3


### Section 8 Observations

- **Acyclic route finding:** Path reconstruction uses list construction alongside cumulative sum evaluation across recursive steps.
- **Graph cycle limitations:** Without tracking visited nodes, recursive traversal is susceptible to infinite loops in cyclic network structures.

# 9. Knowledge check

### Summary of Logic Programming Principles

- **Variables:** Designated by uppercase starting characters or underscores.
- **Unification:** Leverages structural equivalence and argument alignment to bind values.
- **Backtracking:** Procedural mechanics that re-evaluate alternative branches upon subsequent subgoal failure.
- **Base cases:** Essential terminal conditions that ensure bounded recursion halts.
- **Arithmetic:** Evaluated explicitly through arithmetic-evaluation operators rather than standard unification.
- **CUT:** Selectively discards backtracking choices for performance or deterministic routing.
- **Negation as failure:** Represents unprovability within the closed-world schema.
- **Alternative paths:** Backtracking enables the native generation of multiple alternative pathways.

---
# Summary and Reflection

### Declarative Meaning vs. Operational Behavior

- **Declarative design:** Represents the mathematical and logical structures of the problem domains.
- **Operational execution:** Governs the resolution steps, search order, and performance patterns of Prolog programs.